In [1]:
import sys
sys.path.append(r"C:\Users\vinay\OneDrive\Desktop\VATSA\VATSA")
from modules.visual.visual_pipeline import VATSA_VisualPipeline
from modules.visual.streaming import VATSA_VideoStream

pipeline = VATSA_VisualPipeline(
    encoder_path="vatsa_visual_encoder_cifar10_deeper_unfreeze.pth",
    device="cuda"
)

# Test with webcam (source=0) or video file (source="test.mp4")
stream = VATSA_VideoStream(pipeline, source=0, target_fps=15)
benchmark = stream.run(show=True, save_path="vatsa_stream_output.mp4")
print(benchmark)


from modules.visual.benchmark import VATSA_Benchmark

bench = VATSA_Benchmark(pipeline)

bench.run_latency_test(n_runs=50)
bench.run_throughput_test(batch_sizes=[1, 4, 8, 16, 32])
bench.run_embedding_quality_test()
bench.run_memory_test()
bench.print_summary()

report = bench.save_report("vatsa_vmodule_benchmark.json")

Streaming started — press Q to quit

0: 480x640 1 person, 106.4ms
Speed: 7.5ms preprocess, 106.4ms inference, 25.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 15.9ms
Speed: 2.1ms preprocess, 15.9ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 13.2ms
Speed: 1.6ms preprocess, 13.2ms inference, 2.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 13.9ms
Speed: 1.2ms preprocess, 13.9ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 14.5ms
Speed: 1.6ms preprocess, 14.5ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 12.5ms
Speed: 1.4ms preprocess, 12.5ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 13.2ms
Speed: 1.2ms preprocess, 13.2ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 14.6ms
Speed: 1.5ms preprocess, 14.6ms inference, 2.4ms p

- FPS:              22.71
- Embed time:       43.25ms   ← includes YOLO + encoder per frame
- Detections/frame: 3.3 avg

- Detection mean:   18.3ms   p95: 21.4ms
- Embedding mean:   0.0ms    ← synthetic images had no detections so encoder never ran
- Implied FPS:      54.6     ← detection only, no embedding cost

- Batch  1:    89 emb/sec
- Batch  4:   361 emb/sec
- Batch  8:   695 emb/sec
- Batch 16:  1336 emb/sec  ← sweet spot
- Batch 32:  1196 emb/sec  ← VRAM limit, slight drop

- Consistency:          1.0000  ✅ perfect — deterministic
- Augmentation robust:  0.2883  ⚠️  low — expected, encoder trained on classification not contrastive learning
- Inter-image sep:      0.3810  ✅ good — embeddings are distinct

- Allocated:  63.7 MB   ← very lean
- Peak:       73.7 MB
- Reserved:  826.0 MB   ← PyTorch pre-reserves, not actually used

**One note on augmentation robustness (0.2883)**: This is low but not a problem right now. It means the same image cropped differently produces fairly different embeddings. This would matter for contrastive learning (Phase 5 fusion) but is fine for Phase 1 detection. It will improve naturally when we will train with contrastive objectives in Phase 5.

We now have successfully completed. :)
- EfficientNet encoder — 96.31% accuracy
- 512-dim shared latent space
- YOLO detection pipeline
- Bounding box visualisation
- Live streaming with HUD
- Benchmark report saved → vatsa_vmodule_benchmark.json